In [3]:
import pandas as pd
import asyncio
import aiohttp
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, quote

df = pd.read_csv("data.csv")
df["mice"] = False

EMAIL_PROVIDERS = ["gmail.com", "yahoo.com", "ymail.com", "hotmail.com", "outlook.com"]

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

SEM = asyncio.Semaphore(20)  # control concurrency


# -----------------------------
# EMAIL CHECK
# -----------------------------
def is_email_provider(domain):
    if pd.isna(domain):
        return False
    domain = str(domain).lower()
    return any(p in domain for p in EMAIL_PROVIDERS)


# -----------------------------
# FETCH PAGE (ASYNC)
# -----------------------------
async def fetch(session, url):
    try:
        async with SEM:
            async with session.get(url, timeout=10) as response:
                if response.status == 200:
                    return await response.text()
    except:
        return None
    return None


# -----------------------------
# INTERNAL LINKS
# -----------------------------
def get_internal_links(base_url, soup):
    links = set()

    for a in soup.find_all("a", href=True):
        full_url = urljoin(base_url, a["href"])

        if urlparse(full_url).netloc == urlparse(base_url).netloc:
            links.add(full_url)

    return list(links)


# -----------------------------
# CHECK PAGE
# -----------------------------
async def page_has_mice(session, url):
    html = await fetch(session, url)
    if not html:
        return False

    return "mice" in BeautifulSoup(html, "html.parser").get_text(" ", strip=True).lower()


# -----------------------------
# CRAWLER
# -----------------------------
async def crawl_domain(session, domain):
    try:
        base_url = domain if domain.startswith("http") else "https://" + str(domain)

        html = await fetch(session, base_url)
        if not html:
            return False

        soup = BeautifulSoup(html, "html.parser")

        if "mice" in soup.get_text(" ", strip=True).lower():
            return True

        links = get_internal_links(base_url, soup)[:10]

        tasks = [page_has_mice(session, link) for link in links]

        results = await asyncio.gather(*tasks)

        return any(results)

    except:
        return False


# -----------------------------
# SEARCH MODE
# -----------------------------
async def search_and_check(session, company):
    try:
        query = f"{company} MICE meetings incentives conferences exhibitions"
        url = "https://html.duckduckgo.com/html/?q=" + quote(query)

        html = await fetch(session, url)
        if not html:
            return False

        return "mice" in BeautifulSoup(html, "html.parser").get_text(" ", strip=True).lower()

    except:
        return False


# -----------------------------
# PROCESS ROW
# -----------------------------
async def process_row(session, i, row):
    company = row.get("Name Of The Company")
    domain = row.get("domain")

    print(f"[ASYNC] {i} → {company}")

    if is_email_provider(domain):
        found = await search_and_check(session, company)
    else:
        found = await crawl_domain(session, domain)

    return i, found


# -----------------------------
# MAIN
# -----------------------------
async def main():
    async with aiohttp.ClientSession(headers=HEADERS) as session:

        tasks = [
            process_row(session, i, row)
            for i, row in df.iterrows()
        ]

        results = await asyncio.gather(*tasks)

        for i, found in results:
            df.at[i, "mice"] = found

    df.to_csv("data_with_mice.csv", index=False)
    print("\nDONE ✔ Saved file")


# -----------------------------
# RUN
# -----------------------------
await main()

[ASYNC] 0 → 23 DEGREES NORTH
[ASYNC] 1 → 3S Travel Network Pvt Ltd
[ASYNC] 2 → AAKASH GANGA HOLIDAYS
[ASYNC] 3 → AAKRIT TOURISM
[ASYNC] 4 → AARYAN LEISURE & HOLIDAYS PVT LTD
[ASYNC] 5 → ADH TOURS AND TRAVELS PVT. LTD.
[ASYNC] 6 → ADITI TOUR AND TRAVELS
[ASYNC] 7 → ADORE VACATIONS
[ASYNC] 8 → AGWANI TRAVELS PVT LTD.
[ASYNC] 9 → AIR INFINITY TOURS & TRAVELS
[ASYNC] 10 → AIRCOM TRAVELS PVT LTD
[ASYNC] 11 → AIRIXB
[ASYNC] 12 → AKASH TOURISM
[ASYNC] 13 → AKBAR TRAVELS OF INDIA PVT LTD
[ASYNC] 14 → ALL ABOUT HOLIDAYS
[ASYNC] 15 → ALLIANCE AIR AVIATION LTD
[ASYNC] 16 → ALPHA TRAVELS
[ASYNC] 17 → ALPS TOURIST SERVICES PVT LTD
[ASYNC] 18 → ALTAIR SERVICES PVT LTD
[ASYNC] 19 → ALTISGO TOURS AND TRAVEL (OPC) PVT LTD
[ASYNC] 20 → AMBAY TRAVELS ONLINE
[ASYNC] 21 → AMBITION TRAVELS & TOURS PVT LTD
[ASYNC] 22 → AMELIYA TOURISM
[ASYNC] 23 → ANAND WORLD TRAVEL
[ASYNC] 24 → ANANYA TOURISM
[ASYNC] 25 → ANKUR TOUR & TRAVELS PVT. LTD 
[ASYNC] 26 → ANNAPURNA HOLIDAY
[ASYNC] 27 → ANUP TOURS
[ASYNC] 28 → APSA